# Kaggle Titanic Dataset (Random Forest)

In [1]:
# 1) Imports and Paths
import os
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict, RandomizedSearchCV
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance
from sklearn import metrics
import joblib

# Paths and constants
DATA_DIR = "/home/atul-kumar/workspace/kaggle/titanic/data"
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")
SUBMISSION_PATH = os.path.join(DATA_DIR, "submission-rf.csv")
MODEL_DIR = os.path.join(DATA_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Paths set:", TRAIN_PATH, TEST_PATH)

Paths set: /home/atul-kumar/workspace/kaggle/titanic/data/train.csv /home/atul-kumar/workspace/kaggle/titanic/data/test.csv


In [2]:
# 2) Load Data
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
print("Train shape:", train_df.shape, " Test shape:", test_df.shape)
train_df.head(3)

Train shape: (891, 12)  Test shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [3]:
# 3) Feature Engineering Reuse (enhanced)

def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Title from Name and bucket to stable groups
    out["Title"] = out["Name"].str.extract(r",\s*([^\.]+)\.")
    title_map = {
        'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
        'Lady': 'Rare', 'Countess': 'Rare', 'Dona': 'Rare', 'Sir': 'Rare', 'Don': 'Rare',
        'Jonkheer': 'Rare', 'Capt': 'Rare', 'Col': 'Rare', 'Dr': 'Rare', 'Rev': 'Rare',
        'Major': 'Rare'
    }
    out["TitleBucket"] = out["Title"].replace(title_map)
    out.loc[~out["TitleBucket"].isin(['Mr', 'Mrs', 'Miss', 'Master', 'Rare']), "TitleBucket"] = 'Rare'

    # Family features
    out["FamilySize"] = out.get("SibSp", 0) + out.get("Parch", 0) + 1
    out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
    def _family_bin(n):
        if n == 1: return 'Single'
        if 2 <= n <= 4: return 'Small'
        return 'Large'
    out["FamilySizeBin"] = out["FamilySize"].apply(_family_bin)

    # Ticket features
    if "Ticket" in out.columns:
        counts = out["Ticket"].value_counts()
        out["TicketGroup"] = out["Ticket"].map(counts)
        # Letters-only prefix; group rare prefixes
        prefix = out["Ticket"].astype(str).str.replace(r"[^A-Za-z]+", "", regex=True).str.upper()
        prefix = prefix.replace("", np.nan).fillna("NONE")
        pref_counts = prefix.value_counts()
        common = set(pref_counts[pref_counts >= 10].index)
        prefix = prefix.where(prefix.isin(common), other="RARE")
        out["TicketPrefix"] = prefix
    else:
        out["TicketGroup"] = 1
        out["TicketPrefix"] = "NONE"

    # Cabin features
    cabin = out.get("Cabin")
    out["CabinKnown"] = cabin.notna().astype(int)
    out["CabinDeck"] = cabin.astype(str).str[0]
    out["CabinDeck"] = out["CabinDeck"].where(out["CabinKnown"] == 1, other='U')

    # Fare transforms
    out["FareLog"] = np.log1p(out["Fare"]) if "Fare" in out.columns else 0.0

    # Age interactions / bins
    out["AgePclass"] = out.get("Age", np.nan) * out.get("Pclass", np.nan)
    age_bins = [-1, 12, 18, 35, 60, 100]
    age_labels = ["child", "teen", "young", "adult", "senior"]
    out["AgeBand"] = pd.cut(out["Age"], bins=age_bins, labels=age_labels)

    return out

train_df_fe = add_engineered_features(train_df)
test_df_fe = add_engineered_features(test_df)
print("Engineered columns present:", {c for c in train_df_fe.columns if c in [
    "Title","TitleBucket","FamilySize","IsAlone","FamilySizeBin","TicketGroup",
    "TicketPrefix","CabinKnown","CabinDeck","FareLog","AgePclass","AgeBand"
]})

Engineered columns present: {'FareLog', 'TicketPrefix', 'Title', 'CabinDeck', 'IsAlone', 'CabinKnown', 'AgeBand', 'AgePclass', 'FamilySize', 'TitleBucket', 'TicketGroup', 'FamilySizeBin'}


In [4]:
# 4) Define Preprocessing (updated feature lists)
base_features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
engineered = [
    "TitleBucket", "FamilySize", "IsAlone", "FamilySizeBin", "TicketGroup", "TicketPrefix",
    "CabinKnown", "CabinDeck", "FareLog", "AgePclass", "AgeBand"
]
all_features = base_features + engineered

numeric_features = ["Age", "SibSp", "Parch", "Fare", "FamilySize", "TicketGroup", "FareLog", "AgePclass"]
categorical_features = ["Pclass", "Sex", "Embarked", "TitleBucket", "FamilySizeBin", "TicketPrefix", "CabinKnown", "CabinDeck", "AgeBand"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

X = train_df_fe[all_features]
y = train_df_fe["Survived"]
print("Feature matrix shape:", X.shape)

Feature matrix shape: (891, 18)


In [5]:
# 5) Random Forest Model Definition
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    bootstrap=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight=None,
)

model = Pipeline(steps=[
    ("pre", preprocess),
    ("clf", rf),
])
print(model)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Age', 'SibSp', 'Parch',
                                                   'Fare', 'FamilySize',
                                                   'TicketGroup', 'FareLog',
                                                   'AgePclass']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                  

In [18]:
# 6) Cross-Validation Metrics (Accuracy and ROC-AUC)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
acc_scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
auc_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f"CV Accuracy: {acc_scores.mean():.4f} +/- {acc_scores.std():.4f}")
print(f"CV ROC-AUC: {auc_scores.mean():.4f} +/- {auc_scores.std():.4f}")

# Note: AUC = ∫_0^1 TPR(FPR^{-1}(x)) dx
print("AUC is the area under ROC curve: ∫_0^1 TPR(FPR^{-1}(x)) dx")

CV Accuracy: 0.8283 +/- 0.0133
CV ROC-AUC: 0.8760 +/- 0.0252
AUC is the area under ROC curve: ∫_0^1 TPR(FPR^{-1}(x)) dx


In [19]:
# 7) Hyperparameter Search with RandomizedSearchCV
param_distributions = {
    'clf__n_estimators': [300, 600, 900],
    'clf__max_depth': [None, 4, 6, 8, 12],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf': [1, 2, 4],
    'clf__max_features': ['sqrt', 0.6, 0.8],
    'clf__bootstrap': [True, False],
}

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=25,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    refit=True,
    verbose=1,
)

search.fit(X, y)
print("Best params:", search.best_params_)
print("Best ROC-AUC:", round(search.best_score_, 4))

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best params: {'clf__n_estimators': 300, 'clf__min_samples_split': 2, 'clf__min_samples_leaf': 2, 'clf__max_features': 0.6, 'clf__max_depth': 6, 'clf__bootstrap': True}
Best ROC-AUC: 0.8856
Best params: {'clf__n_estimators': 300, 'clf__min_samples_split': 2, 'clf__min_samples_leaf': 2, 'clf__max_features': 0.6, 'clf__max_depth': 6, 'clf__bootstrap': True}
Best ROC-AUC: 0.8856


In [20]:
# 8) Refit Best Model on Full Training Data
best_estimator = search.best_estimator_
best_estimator.fit(X, y)
final_model = best_estimator
print("Best model refit on full data.")

Best model refit on full data.


In [21]:
# 9) Probability Calibration (Optional)
calibrated_model = CalibratedClassifierCV(estimator=final_model, method='sigmoid', cv=5)
# Compare CV log loss between final_model and calibrated_model
logloss_cv_final = -cross_val_score(final_model, X, y, cv=cv, scoring='neg_log_loss', n_jobs=-1)
logloss_cv_calibrated = -cross_val_score(calibrated_model, X, y, cv=cv, scoring='neg_log_loss', n_jobs=-1)
print(f"LogLoss final: {logloss_cv_final.mean():.4f} +/- {logloss_cv_final.std():.4f}")
print(f"LogLoss calibrated: {logloss_cv_calibrated.mean():.4f} +/- {logloss_cv_calibrated.std():.4f}")

chosen_model = calibrated_model if logloss_cv_calibrated.mean() < logloss_cv_final.mean() else final_model
print("Chosen model:", "calibrated" if chosen_model is calibrated_model else "final (uncalibrated)")

LogLoss final: 0.3950 +/- 0.0253
LogLoss calibrated: 0.3952 +/- 0.0219
Chosen model: final (uncalibrated)


In [22]:
# 10) Threshold Tuning on Validation Predictions
# Get out-of-fold probabilities for chosen_model
p_oof = cross_val_predict(chosen_model, X, y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]

best_threshold = 0.5
best_f1 = -1
for t in np.linspace(0.2, 0.8, 121):
    preds = (p_oof >= t).astype(int)
    f1 = metrics.f1_score(y, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = float(t)

acc = metrics.accuracy_score(y, (p_oof >= best_threshold).astype(int))
print(f"Best threshold: {best_threshold:.3f} with F1={best_f1:.4f}, Accuracy={acc:.4f}")

Best threshold: 0.415 with F1=0.7792, Accuracy=0.8283


In [23]:
# 11) Permutation Feature Importance (per original input features)
from sklearn.model_selection import train_test_split

# Holdout split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Use the chosen_model directly so permutation operates on raw input columns
chosen_model.fit(X_train, y_train)

r = permutation_importance(
    chosen_model, X_valid, y_valid,
    scoring='roc_auc', n_repeats=10,
    random_state=RANDOM_STATE, n_jobs=-1
)

feature_names = list(all_features)
importances = pd.Series(r.importances_mean, index=feature_names).sort_values(ascending=False)
print("Top 15 features by permutation importance (input level):")
importances.head(15)

Top 15 features by permutation importance (input level):


Sex              0.064717
TitleBucket      0.055817
Pclass           0.027892
AgePclass        0.013360
Age              0.009387
TicketGroup      0.008432
FamilySizeBin    0.006456
FamilySize       0.005889
Fare             0.003030
Embarked         0.002490
FareLog          0.002279
TicketPrefix     0.001884
AgeBand          0.001726
SibSp            0.000487
Parch            0.000211
dtype: float64

In [24]:
# 12) Predict Test Set and Save Submission
X_test = test_df_fe[all_features]
probs_test = chosen_model.predict_proba(X_test)[:, 1]
labels_test = (probs_test >= best_threshold).astype(int)

submission = pd.DataFrame({
    'PassengerId': test_df_fe['PassengerId'],
    'Survived': labels_test
})
submission.to_csv(SUBMISSION_PATH, index=False)
print("Saved submission to:", SUBMISSION_PATH)
submission.head(10)

Saved submission to: /home/atul-kumar/workspace/kaggle/titanic/data/submission-rf.csv


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0


In [13]:
# 13) Save Trained Model Artifact
metadata = {
    'best_params': getattr(search, 'best_params_', None),
    'best_threshold': best_threshold,
    'cv_accuracy': float(np.mean(acc_scores)),
    'cv_auc': float(np.mean(auc_scores)),
}
artifact_path = os.path.join(MODEL_DIR, 'random-forest-titanic.joblib')
joblib.dump({'model': chosen_model, 'metadata': metadata}, artifact_path)
print("Saved model artifact to:", artifact_path)
metadata

Saved model artifact to: /home/atul-kumar/workspace/kaggle/titanic/data/models/random-forest-titanic.joblib


{'best_params': {'clf__n_estimators': 900,
  'clf__min_samples_split': 5,
  'clf__min_samples_leaf': 1,
  'clf__max_features': 0.6,
  'clf__max_depth': 12,
  'clf__bootstrap': True},
 'best_threshold': 0.49000000000000005,
 'cv_accuracy': 0.8024606113866047,
 'cv_auc': 0.8711836287983663}

In [14]:
# 14) Reproducibility: Random Seeds and Determinism
# Already set RANDOM_STATE and numpy seed. For extra determinism in terminal runs:
os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)
os.environ.setdefault('OMP_NUM_THREADS', '1')
print("Seeds and environment set for reproducibility.")

Seeds and environment set for reproducibility.


## RF v2: Refined features + broader tuning

We trim noisy engineered features, add a high-signal ratio (FarePerPerson), and broaden the search space (criterion, class_weight, n_estimators, max_samples). This section builds a second RF pipeline without altering your original results.

In [6]:
# v2.1) Feature engineering adjustments (for RF only)
# - Drop: TicketPrefix, CabinDeck, AgeBand
# - Add: FarePerPerson

def add_rf_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # Reuse existing engineered columns if present; otherwise recompute minimal ones
    if 'FamilySize' not in out.columns:
        out['FamilySize'] = out.get('SibSp', 0) + out.get('Parch', 0) + 1
    if 'IsAlone' not in out.columns:
        out['IsAlone'] = (out['FamilySize'] == 1).astype(int)
    if 'Title' not in out.columns or 'TitleBucket' not in out.columns:
        out['Title'] = out['Name'].str.extract(r",\s*([^\.]+)\.")
        title_map = {
            'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
            'Lady': 'Rare', 'Countess': 'Rare', 'Dona': 'Rare', 'Sir': 'Rare', 'Don': 'Rare',
            'Jonkheer': 'Rare', 'Capt': 'Rare', 'Col': 'Rare', 'Dr': 'Rare', 'Rev': 'Rare',
            'Major': 'Rare'
        }
        out['TitleBucket'] = out['Title'].replace(title_map)
        out.loc[~out['TitleBucket'].isin(['Mr','Mrs','Miss','Master','Rare']), 'TitleBucket'] = 'Rare'
    # TicketGroup (size)
    if 'TicketGroup' not in out.columns:
        if 'Ticket' in out.columns:
            counts = out['Ticket'].value_counts()
            out['TicketGroup'] = out['Ticket'].map(counts)
        else:
            out['TicketGroup'] = 1
    # FareLog and AgePclass (reuse or compute)
    if 'FareLog' not in out.columns:
        out['FareLog'] = np.log1p(out['Fare']) if 'Fare' in out.columns else 0.0
    if 'AgePclass' not in out.columns:
        out['AgePclass'] = out.get('Age', np.nan) * out.get('Pclass', np.nan)
    # New: FarePerPerson
    with np.errstate(divide='ignore', invalid='ignore'):
        out['FarePerPerson'] = out['Fare'] / out['FamilySize']
    return out

train_rf = add_rf_features(train_df.copy())
test_rf = add_rf_features(test_df.copy())

base = ["Pclass","Sex","Age","SibSp","Parch","Fare","Embarked"]
engineered_rf = [
    "TitleBucket","FamilySize","IsAlone","TicketGroup","FareLog","AgePclass","FarePerPerson"
]
features_rf = base + engineered_rf

num_rf = ["Age","SibSp","Parch","Fare","FamilySize","TicketGroup","FareLog","AgePclass","FarePerPerson"]
cat_rf = ["Pclass","Sex","Embarked","TitleBucket","IsAlone"]

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

num_tf = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
cat_tf = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
pre_rf = ColumnTransformer([
    ("num", num_tf, num_rf),
    ("cat", cat_tf, cat_rf),
])

X_rf = train_rf[features_rf]
y_rf = train_rf['Survived']
X_test_rf = test_rf[features_rf]
print("RF v2 feature matrix:", X_rf.shape)


RF v2 feature matrix: (891, 14)


In [7]:
# v2.2) RF model + broader hyperparameter search
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_score

rf_v2 = Pipeline(steps=[
    ("pre", pre_rf),
    ("clf", RandomForestClassifier(random_state=63))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=63)

param_dist = {
    'clf__n_estimators': [200, 300, 400, 600, 800],
    'clf__max_depth': [None, 4, 6, 8, 10],
    'clf__min_samples_leaf': [1, 2, 3, 4],
    'clf__min_samples_split': [2, 4, 6, 8],
    'clf__max_features': [0.4, 0.6, 0.8, 'sqrt', 'log2'],
    'clf__bootstrap': [True],
    'clf__criterion': ['gini', 'entropy', 'log_loss'],
    'clf__class_weight': [None, 'balanced'],
    'clf__max_samples': [None, 0.8, 0.9]
}

search_v2 = RandomizedSearchCV(
    estimator=rf_v2,
    param_distributions=param_dist,
    n_iter=60,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    random_state=63,
    refit=True,
    verbose=1,
)

print("Fitting RF v2 RandomizedSearch...")
search_v2.fit(X_rf, y_rf)
print("Best params:", search_v2.best_params_)
print("Best ROC-AUC:", round(search_v2.best_score_, 4))

best_rf_v2 = search_v2.best_estimator_
acc = cross_val_score(best_rf_v2, X_rf, y_rf, cv=cv, scoring='accuracy', n_jobs=-1)
auc = cross_val_score(best_rf_v2, X_rf, y_rf, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f"RF v2 CV Accuracy: {acc.mean():.4f} +/- {acc.std():.4f}")
print(f"RF v2 CV ROC-AUC: {auc.mean():.4f} +/- {auc.std():.4f}")


Fitting RF v2 RandomizedSearch...
Fitting 5 folds for each of 60 candidates, totalling 300 fits
Best params: {'clf__n_estimators': 400, 'clf__min_samples_split': 6, 'clf__min_samples_leaf': 1, 'clf__max_samples': 0.8, 'clf__max_features': 0.4, 'clf__max_depth': 6, 'clf__criterion': 'gini', 'clf__class_weight': None, 'clf__bootstrap': True}
Best ROC-AUC: 0.8769
Best params: {'clf__n_estimators': 400, 'clf__min_samples_split': 6, 'clf__min_samples_leaf': 1, 'clf__max_samples': 0.8, 'clf__max_features': 0.4, 'clf__max_depth': 6, 'clf__criterion': 'gini', 'clf__class_weight': None, 'clf__bootstrap': True}
Best ROC-AUC: 0.8769
RF v2 CV Accuracy: 0.8328 +/- 0.0144
RF v2 CV ROC-AUC: 0.8769 +/- 0.0149
RF v2 CV Accuracy: 0.8328 +/- 0.0144
RF v2 CV ROC-AUC: 0.8769 +/- 0.0149


In [8]:
# v2.3) Fit best RF v2 on full data, tune threshold, predict test & save submission
from sklearn.model_selection import cross_val_predict
from sklearn import metrics

best_rf_v2.fit(X_rf, y_rf)

# OOF probabilities for threshold tuning
p_oof_v2 = cross_val_predict(best_rf_v2, X_rf, y_rf, cv=cv, method='predict_proba', n_jobs=-1)[:,1]

best_t = 0.5
best_acc = -1
for t in np.linspace(0.3, 0.7, 81):
    acc = metrics.accuracy_score(y_rf, (p_oof_v2 >= t).astype(int))
    if acc > best_acc:
        best_acc = acc
        best_t = float(t)
print(f"Best threshold (RF v2): {best_t:.3f} with ACC={best_acc:.4f}")

# Predict test
probs_test = best_rf_v2.predict_proba(X_test_rf)[:,1]
labels_test = (probs_test >= best_t).astype(int)

SUBMISSION_PATH_V2 = os.path.join("/home/atul-kumar/workspace/kaggle/titanic/data", "submission-rf-v2.csv")
submission = pd.DataFrame({ 'PassengerId': test_rf['PassengerId'], 'Survived': labels_test })
submission.to_csv(SUBMISSION_PATH_V2, index=False)
print("Saved RF v2 submission to:", SUBMISSION_PATH_V2)
submission.head(10)


Best threshold (RF v2): 0.535 with ACC=0.8418
Saved RF v2 submission to: /home/atul-kumar/workspace/kaggle/titanic/data/submission-rf-v2.csv


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,0
7,899,0
8,900,1
9,901,0


## RF v2 (accuracy-optimized)
We run a focused search optimizing cross-validated accuracy directly and then tune the classification threshold on OOF probabilities.

In [9]:
# v2.4) Accuracy-optimized search and submission
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict

rf_acc = Pipeline(steps=[("pre", pre_rf), ("clf", RandomForestClassifier(random_state=63))])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=63)

param_dist_acc = {
    'clf__n_estimators': [300, 400, 500, 700],
    'clf__max_depth': [None, 6, 8, 10],
    'clf__min_samples_leaf': [1, 2, 3, 4],
    'clf__min_samples_split': [2, 4, 6, 8],
    'clf__max_features': [0.3, 0.4, 0.6, 'sqrt'],
    'clf__bootstrap': [True],
    'clf__class_weight': [None, 'balanced_subsample'],
    'clf__max_samples': [None, 0.75, 0.85]
}

search_acc = RandomizedSearchCV(
    estimator=rf_acc,
    param_distributions=param_dist_acc,
    n_iter=50,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
    random_state=63,
    refit=True,
    verbose=1,
)

print("Fitting RF v2 (accuracy-optimized) RandomizedSearch...")
search_acc.fit(X_rf, y_rf)
print("Best params (ACC-optimized):", search_acc.best_params_)
print("Best CV Accuracy:", round(search_acc.best_score_, 4))

rf_acc_best = search_acc.best_estimator_
# OOF for threshold tuning (still tune, even if optimizing accuracy)
p_acc_oof = cross_val_predict(rf_acc_best, X_rf, y_rf, cv=cv, method='predict_proba', n_jobs=-1)[:,1]
best_t = 0.5
best_acc = -1
for t in np.linspace(0.35, 0.65, 61):
    acc_t = metrics.accuracy_score(y_rf, (p_acc_oof >= t).astype(int))
    if acc_t > best_acc:
        best_acc = acc_t
        best_t = float(t)
print(f"Best threshold (ACC-optimized RF): {best_t:.3f} with ACC={best_acc:.4f}")

# Predict test & save
p_test = rf_acc_best.predict_proba(X_test_rf)[:,1]
labels_test = (p_test >= best_t).astype(int)
SUBMISSION_PATH_V2_ACC = os.path.join(DATA_DIR, "submission-rf-v2-acc.csv")
submission = pd.DataFrame({ 'PassengerId': test_rf['PassengerId'], 'Survived': labels_test })
submission.to_csv(SUBMISSION_PATH_V2_ACC, index=False)
print("Saved accuracy-optimized RF v2 submission to:", SUBMISSION_PATH_V2_ACC)
submission.head(10)

Fitting RF v2 (accuracy-optimized) RandomizedSearch...
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params (ACC-optimized): {'clf__n_estimators': 700, 'clf__min_samples_split': 4, 'clf__min_samples_leaf': 2, 'clf__max_samples': 0.75, 'clf__max_features': 0.6, 'clf__max_depth': 6, 'clf__class_weight': None, 'clf__bootstrap': True}
Best CV Accuracy: 0.8361
Best threshold (ACC-optimized RF): 0.525 with ACC=0.8384
Saved accuracy-optimized RF v2 submission to: /home/atul-kumar/workspace/kaggle/titanic/data/submission-rf-v2-acc.csv


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,0
7,899,0
8,900,1
9,901,0
